In [ ]:
''' 
A screener to check if the tickers meet the graham guidlines and are below the 200 day sma 
I have made the changes
further changes
making some more changes
more fucking changes


'''

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import time 

In [ ]:
# asx tickers to run through the screener


asxtickers = pd.read_csv('ASX_Listed_Companies_26-01-2026_10-34-23_AEDT.csv')
asxtickers = asxtickers['ASX code']
asxtickers = asxtickers + '.ax'
asxtickers.head()


In [ ]:
def metrics(ticker):
    try:
        t = yf.Ticker(ticker)
        info = t.info
        hist = t.history(period='1y')

        if len(hist) < 200:
            return None

        sma200 = hist['Close'].rolling(200).mean().iloc[-1]
        current_price = hist['Close'].iloc[-1]

        eps = info.get('trailingEps', 0)
        bvps = info.get('bookValue', 0)
        graham_number = np.sqrt(22.5 * eps * bvps) if eps > 0 and bvps > 0 else 0

        # Safely get and convert P/E, P/B, Debt/Eq to numeric
        pe = info.get('trailingPE')
        pb = info.get('priceToBook')
        debt_eq = info.get('debtToEquity')

        try:
            pe = float(pe) if pe is not None else None
        except ValueError:
            pe = None

        try:
            pb = float(pb) if pb is not None else None
        except ValueError:
            pb = None

        try:
            debt_eq = float(debt_eq) if debt_eq is not None else None
        except ValueError:
            debt_eq = None

        # small delay to avoid rate limiting
        time.sleep(0.5) 

        return {
            'Ticker': ticker,
            'Price': current_price,
            'SMA200': sma200,
            'Below_SMA': current_price < sma200,
            'P/E': pe,
            'P/B': pb,
            'Debt/Eq': debt_eq,
            'Current_Ratio': info.get('currentRatio'),
            'Div_Yield': info.get('dividendYield', 0) * 100,
            'EPS': eps,
            'ROE': info.get('returnOnEquity', 0) * 100,
            'Graham_Number': graham_number,
            'MoS_%': ((graham_number / current_price - 1) * 100) if current_price > 0 else 0
        }
    except Exception as e:
        # Print the error but return None so screening can continue
        print(f"Error fetching {ticker}: {e}")
        time.sleep(1) # Longer delay on error to cool down
        return None

In [ ]:
def screener(tickers, pe_max=15, pb_max=1.5, debt_max=50, div_min=2, sma_discount=0):
  print(f'Screening {len(tickers)} tickers.')
  results = []

  for i, ticker in enumerate(tickers, 1):
    print(f'Screening {i}/{len(tickers)}: {ticker}')
    metrics_data = metrics(ticker)

    if (metrics_data and
            metrics_data['P/E'] is not None and metrics_data['P/E'] > 0 and metrics_data['P/E'] < pe_max and
            metrics_data['P/B'] is not None and metrics_data['P/B'] > 0 and metrics_data['P/B'] < pb_max and
            metrics_data['Debt/Eq'] is not None and metrics_data['Debt/Eq'] > 0 and metrics_data['Debt/Eq'] < debt_max and
            metrics_data['Div_Yield'] > div_min and
            metrics_data['Price'] < metrics_data['SMA200'] * (1 - sma_discount/100)
        ):
            results.append(metrics_data)
            print(f"  ✓ {ticker} PASSED!")

  if not results:
      print("No candidates found. Try relaxing criteria.")
      return pd.DataFrame(columns=['Ticker', 'Price', 'P/E', 'P/B', 'Div_Yield', 'MoS_%', 'Graham_Number'])

  df = pd.DataFrame(results).sort_values('Graham_Number', ascending=False)
  print(f"\n✅ Found {len(df)} Graham candidates!")
  return df

In [ ]:
if __name__ == "__main__":
    # RELAXED screen first time
    print("=== RELAXED GRAHAM SCREEN (ASX Tickers 601-1200) ===")
    candidates_relaxed = screener(  # FIXED: function name + variable
        asxtickers.iloc[1201:1855],
        pe_max=20, pb_max=2.5, debt_max=100, div_min=1, sma_discount=10
    )

    print("\n=== STRICT GRAHAM SCREEN (FROM RELAXED CANDIDATES) ===")
    candidates_strict = screener(  # Use relaxed candidates for strict screening
        candidates_relaxed['Ticker'].tolist(), # Pass only tickers from relaxed candidates
        pe_max=12, pb_max=1.5, debt_max=50, div_min=2, sma_discount=5
    )

    # Display and export (INDENT FIXED - now inside if __name__)
    def show_results(df, title):
        if not df.empty:
            print(f"\n{title}")
            print(df[['Ticker', 'Price', 'P/E', 'P/B', 'Div_Yield', 'MoS_%']].round(2))
            filename = f"{title.lower().replace(' ', '_').replace(':', '')}.csv"
            df.to_csv(filename, index=False)
            files.download(filename)
            print(f"💾 Downloaded: {filename}")
        else:
            print(f"{title}: No candidates")

    show_results(candidates_relaxed, "RELAXED SCREEN RESULTS (601-1200)")
    show_results(candidates_strict, "STRICT GRAHAM RESULTS (from 601-1200 relaxed)")

    # Quick summary
    print("\n📊 SUMMARY (ASX Tickers 601-1200 SCAN):")
    print(f"Relaxed hits: {len(candidates_relaxed)} | Strict hits: {len(candidates_strict)}")

In [ ]:
#rejig thos thing


